In [ ]:
import pandas as pd
import numpy as np
import pydicom
from tqdm.auto import tqdm
from PIL import Image
from pathlib import Path
from fastai.vision.all import *
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
TRAIN_SERIES_DIR = DATA / 'train_series'
TEST_SERIES_DIR  = DATA / 'test_series'
WORKING = Path('/kaggle/working')
SLICE_DIR = WORKING / 'middle_slices'
SLICE_DIR.mkdir(exist_ok=True)

In [ ]:
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
          'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis',
          "Baker's", 'Contusion', 'Fracture']

In [ ]:
def read_dicom(path):
    ds = pydicom.dcmread(path, force=True)
    img = ds.pixel_array.astype(np.float32)
    if hasattr(ds, 'RescaleSlope') and hasattr(ds, 'RescaleIntercept'):
        img = img * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

def get_largest_series(study_dir: Path):
    series_dirs = [p for p in study_dir.iterdir() if p.is_dir()]
    if not series_dirs:
        return None
    return max(series_dirs, key=lambda p: len(list(p.glob('*.dcm'))))

def extract_middle_slice(study_uid: str, series_root: Path, out_dir: Path = None, save: bool = True):
    study_dir = series_root / study_uid
    if not study_dir.exists():
        return None
    sdir = get_largest_series(study_dir)
    if sdir is None:
        return None
    dcms = sorted(list(sdir.glob('*.dcm')), key=lambda x: int(x.stem) if x.stem.isdigit() else 0)
    if not dcms:
        return None
    mid = len(dcms) // 2
    img = read_dicom(dcms[mid])
    if save and out_dir is not None:
        out_path = out_dir / f"{study_uid}.png"
        Image.fromarray((img * 255).astype(np.uint8)).save(out_path)
        return out_path
    return img

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
labeled = train.dropna(subset=LABELS).copy()
labeled[LABELS] = labeled[LABELS].fillna(0).astype(int)

In [ ]:
rows = []
for uid in tqdm(labeled['StudyInstanceUID'], desc='Extracting train slices'):
    img_path = extract_middle_slice(uid, TRAIN_SERIES_DIR, SLICE_DIR, save=True)
    if img_path is not None:
        row = {'StudyInstanceUID': uid, 'image_path': str(img_path)}
        for lab in LABELS:
            row[lab] = labeled.loc[labeled['StudyInstanceUID'] == uid, lab].values[0]
        rows.append(row)

In [ ]:
df_slices = pd.DataFrame(rows)
print(f"Train slices ready: {len(df_slices)}")

In [ ]:
def get_y(row):
    return [lab for lab in LABELS if row[lab] == 1]

dblock = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock(vocab=LABELS)),
    get_x=ColReader('image_path'),
    get_y=get_y,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=Resize(384, method=ResizeMethod.Pad, pad_mode='zeros'),
    batch_tfms=[
        *aug_transforms(size=224, max_rotate=10, max_zoom=1.1,
                        max_lighting=0.2, p_affine=0.75, p_lighting=0.75),
        Normalize.from_stats(*imagenet_stats)
    ]
)

In [ ]:
dls = dblock.dataloaders(df_slices, bs=32, num_workers=2)

In [ ]:
dls.show_batch(max_n=9)

In [ ]:
learn = vision_learner(
    dls,
    resnet34,
    metrics=[accuracy_multi, RocAucMulti(average='macro')],
    loss_func=BCEWithLogitsLossFlat()
)

In [ ]:
learn.lr_find()

In [ ]:
learn.fine_tune(5, base_lr=1e-3)

In [ ]:
#save model
learn.export(WORKING / 'baseline_export.pkl')

## Submission

In [ ]:
test = pd.read_csv(DATA / 'test.csv') 
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
test_uids = sample_sub['StudyInstanceUID'].unique().tolist()

In [ ]:
TEST_SLICE_DIR = WORKING / 'test_middle_slices'
TEST_SLICE_DIR.mkdir(exist_ok=True)

In [ ]:
test_rows = []
for uid in tqdm(test_uids, desc='Extracting test slices'):
    img_path = extract_middle_slice(uid, TEST_SERIES_DIR, TEST_SLICE_DIR, save=True)
    if img_path is not None:
        test_rows.append({'StudyInstanceUID': uid, 'image_path': str(img_path)})

In [ ]:
df_test = pd.DataFrame(test_rows)
print(f"Test slices ready: {len(df_test)}")

In [ ]:
test_dl = learn.dls.test_dl(df_test)

In [ ]:
preds, _ = learn.get_preds(dl=test_dl)
preds = preds.numpy()

In [ ]:
sub = pd.DataFrame(preds, columns=LABELS)
sub.insert(0, 'StudyInstanceUID', df_test['StudyInstanceUID'].values)

In [ ]:
sub = sub[sample_sub.columns]

In [ ]:
missing = set(sample_sub['StudyInstanceUID']) - set(sub['StudyInstanceUID'])
if missing:
    print(f"Missing {len(missing)} studies – filling with 0.5")
    fill = pd.DataFrame({'StudyInstanceUID': list(missing)})
    for c in LABELS:
        fill[c] = 0.5
    sub = pd.concat([sub, fill], ignore_index=True)

In [ ]:
sub = sub.set_index('StudyInstanceUID').loc[sample_sub['StudyInstanceUID']].reset_index()
sub.to_csv(WORKING / 'submission.csv', index=False)
print("Saved submission.csv")
print(sub.head())
print(sub.shape)

In [ ]:
# !kaggle competitions submit -c rsna-knee-abnormality-detection -f /kaggle/working/submission.csv -m "Baseline: Resnet34 https://www.kaggle.com/code/utkarshtomar736/rsna-baseline-fastai"